# Summa Mathematica — Colab OCR pipeline (PDF → raw Markdown), resumable

Part 1 of the pipeline. **OCR only**: `marker-pdf` converts the PDFs to Markdown, and the result is pushed to
`raw_ocr/` of the **private** repository `math_books`. No LLM / DeepSeek API is called here.

**Built for large folders (hundreds of books) and interrupted sessions:**
- The Drive folder is only *listed*; each PDF is downloaded just before it is processed and deleted afterwards.
- Each book (and each chunk of `CHUNK_PAGES` pages) is OCR'd in a **separate Python process**, so the memory used by
  marker's models is returned to the system after every book.
- Progress is kept in `raw_ocr/_manifest.json` in `math_books`. After every book the Markdown and the manifest are
  committed and pushed together. Before a book starts, its attempt count is pushed, so a book that crashes the
  runtime is retried at most `MAX_ATTEMPTS` times and then skipped.
- **To resume after a disconnect or crash: Runtime → Run all.** Finished books are skipped.

Before the first run:
1. Runtime → Change runtime type → **T4 GPU** (CPU works but is very slow).
2. Colab secrets (🔑 in the left bar) → add `GITHUB_TOKEN`: a fine-grained personal access token with
   *Contents: Read and write* on `sirisaacrewton-glitch/math_books` only. Enable “Notebook access”.
3. Colab secrets → add `DRIVE_FOLDER_ID`: the ID of the Google Drive folder holding the PDFs
   (the part after `/folders/` in its URL). It is kept out of this notebook because the notebook is public.

Book text must never be pushed to the public repository `summa-mathematica`; this notebook only writes to `math_books`.


In [ ]:
# ===== Settings =====
from google.colab import userdata
DRIVE_FOLDER_ID = userdata.get("DRIVE_FOLDER_ID").strip()   # Colab secret (not written here: this file is public)

# How to reach the PDFs:
#   "api"   : sign in with google.colab.auth and use the Drive API (private and shared folders; recommended)
#   "mount" : mount your Google Drive (the folder must be in My Drive or added to it as a shortcut)
SOURCE_MODE = "api"      # "mount" falls back to "api" if mounting fails
DRIVE_FOLDER_PATH = ""   # mount only, optional: e.g. "/content/drive/MyDrive/math_pdfs"

GITHUB_REPO   = "sirisaacrewton-glitch/math_books"   # PRIVATE repository (book text allowed)
GITHUB_BRANCH = "main"
OUTPUT_DIR    = "raw_ocr"                              # inside the repository
MANIFEST      = "raw_ocr/_manifest.json"               # progress file (in the repository)

ONLY_FILES    = []      # e.g. ["klingenberg-diffgeom.pdf"]; empty = every PDF in the folder
FORCE_OCR     = False   # True: OCR every page even if the PDF has a text layer (scanned books)
CHUNK_PAGES   = 120     # pages per OCR process (smaller = less memory; 0 = whole book in one process)
BOOK_TIMEOUT_MIN = 240  # give up on a book after this many minutes
MAX_ATTEMPTS  = 2       # a book that failed or crashed this many times is skipped (status "failed")
RETRY_FAILED  = False   # True: try "failed" books again
MAX_BOOKS     = 0       # stop after this many books in this run (0 = no limit)

In [ ]:
# ===== Install (takes a few minutes; restart is not needed) =====
!pip -q install "marker-pdf>=1.8" pypdf
import torch, importlib.metadata as md
print("marker-pdf", md.version("marker-pdf"), "| CUDA:", torch.cuda.is_available())

In [ ]:
# ===== Clone the private repository and load the manifest =====
import os, json, shutil, subprocess, time, datetime, re, pathlib
from google.colab import userdata
TOKEN = userdata.get("GITHUB_TOKEN")
REPO_DIR = "/content/repo"

def run(cmd, cwd=None):
    # never echo the token
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(("command failed: %s\n%s" % (" ".join(cmd), r.stderr)).replace(TOKEN, "***"))
    return r.stdout

if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
remote = "https://x-access-token:%s@github.com/%s.git" % (TOKEN, GITHUB_REPO)
run(["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, remote, REPO_DIR])
run(["git", "config", "user.name", "colab-ocr"], cwd=REPO_DIR)
run(["git", "config", "user.email", "colab-ocr@users.noreply.github.com"], cwd=REPO_DIR)
os.makedirs(os.path.join(REPO_DIR, OUTPUT_DIR), exist_ok=True)

def load_manifest():
    p = os.path.join(REPO_DIR, MANIFEST)
    if os.path.exists(p):
        return json.load(open(p, encoding="utf-8"))
    return {"version": 1, "books": {}}

def save_manifest(m):
    with open(os.path.join(REPO_DIR, MANIFEST), "w", encoding="utf-8") as f:
        json.dump(m, f, ensure_ascii=False, indent=1, sort_keys=True)

def push(paths, message):
    # commit the given files and push; on a conflict, rebase on the remote and retry
    run(["git", "add", *paths], cwd=REPO_DIR)
    if subprocess.run(["git", "diff", "--cached", "--quiet"], cwd=REPO_DIR).returncode == 0:
        return
    run(["git", "commit", "-m", message], cwd=REPO_DIR)
    for i in range(5):
        try:
            run(["git", "pull", "--rebase", "-X", "theirs", "origin", GITHUB_BRANCH], cwd=REPO_DIR)
            run(["git", "push", "origin", "HEAD:" + GITHUB_BRANCH], cwd=REPO_DIR)
            return
        except RuntimeError as e:
            print("push failed, retrying:", str(e)[:200])
            time.sleep(2 ** (i + 1))
    raise RuntimeError("push failed 5 times")

manifest = load_manifest()
c = {}
for b in manifest["books"].values():
    c[b["status"]] = c.get(b["status"], 0) + 1
print("cloned", GITHUB_REPO, "| manifest:", c or "empty")

In [ ]:
# ===== List the PDFs (no download yet) =====
import glob, logging
BOOKS = []   # [{"key", "name", "drive_id" or "path"}]

if SOURCE_MODE == "mount":
    from google.colab import drive
    try:
        drive.mount("/content/drive")
    except Exception as e:  # e.g. MessageError: credential propagation was unsuccessful
        print("drive.mount failed (%s); falling back to SOURCE_MODE = 'api'" % e)
        SOURCE_MODE = "api"

if SOURCE_MODE == "api":
    # harmless notice from the auth library ("httplib2 transport does not support per-request timeout")
    logging.getLogger("google_auth_httplib2").setLevel(logging.ERROR)
    from google.colab import auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload
    auth.authenticate_user()
    service = build("drive", "v3")
    def children(fid):
        token = None
        while True:
            r = service.files().list(q="'%s' in parents and trashed = false" % fid,
                                     fields="nextPageToken, files(id, name, mimeType, shortcutDetails)",
                                     pageSize=1000, pageToken=token,
                                     supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
            yield from r.get("files", [])
            token = r.get("nextPageToken")
            if not token:
                return
    stack = [DRIVE_FOLDER_ID]
    while stack:
        for f in children(stack.pop()):
            fid, mime = f["id"], f["mimeType"]
            if mime == "application/vnd.google-apps.shortcut":
                fid, mime = f["shortcutDetails"]["targetId"], f["shortcutDetails"]["targetMimeType"]
            if mime == "application/vnd.google-apps.folder":
                stack.append(fid)
            elif f["name"].lower().endswith(".pdf"):
                BOOKS.append({"key": fid, "name": f["name"], "drive_id": fid})
else:
    def find_folder_by_id(root, folder_id):
        for dirpath, dirnames, _ in os.walk(root):
            for d in dirnames:
                p = os.path.join(dirpath, d)
                try:
                    if os.getxattr(p, "user.drive.id").decode() == folder_id:
                        return p
                except OSError:
                    pass
        return None
    src = DRIVE_FOLDER_PATH if DRIVE_FOLDER_PATH and os.path.isdir(DRIVE_FOLDER_PATH) else \
          find_folder_by_id("/content/drive/MyDrive", DRIVE_FOLDER_ID) or \
          find_folder_by_id("/content/drive/Shareddrives", DRIVE_FOLDER_ID)
    if src is None:
        raise SystemExit("Folder not found in My Drive: add a shortcut to it in My Drive, set DRIVE_FOLDER_PATH, "
                         "or use SOURCE_MODE = 'api'.")
    for p in glob.glob(os.path.join(src, "**", "*.pdf"), recursive=True):
        BOOKS.append({"key": os.path.relpath(p, src), "name": os.path.basename(p), "path": p})

if ONLY_FILES:
    BOOKS = [b for b in BOOKS if b["name"] in ONLY_FILES]
BOOKS.sort(key=lambda b: b["name"].lower())
print(len(BOOKS), "PDF(s) in the folder")

In [ ]:
# ===== The OCR worker (runs in its own process for each chunk) =====
WORKER = "/content/ocr_worker.py"
open(WORKER, "w").write(r'''
import sys, json
pdf, out, page_range, force_ocr = sys.argv[1], sys.argv[2], sys.argv[3], sys.argv[4] == "1"
from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.output import text_from_rendered
from marker.config.parser import ConfigParser
options = {"output_format": "markdown", "disable_image_extraction": True, "force_ocr": force_ocr}
if page_range != "all":
    options["page_range"] = page_range
config = ConfigParser(options)
converter = PdfConverter(artifact_dict=create_model_dict(), config=config.generate_config_dict(),
                         processor_list=config.get_processors(), renderer=config.get_renderer(),
                         llm_service=None)
text, _, _ = text_from_rendered(converter(pdf))
open(out, "w", encoding="utf-8").write(text)
print(json.dumps({"chars": len(text)}))
''')

from pypdf import PdfReader
PDF_DIR = "/content/pdf_work"

def out_name(name):
    return re.sub(r"[^A-Za-z0-9._-]+", "-", pathlib.Path(name).stem).strip("-") + ".md"

def fetch(book):
    # copy or download one PDF to the work folder
    shutil.rmtree(PDF_DIR, ignore_errors=True)
    os.makedirs(PDF_DIR)
    dst = os.path.join(PDF_DIR, "book.pdf")
    if "path" in book:
        shutil.copy2(book["path"], dst)
    else:
        with open(dst, "wb") as fh:
            dl = MediaIoBaseDownload(fh, service.files().get_media(fileId=book["drive_id"], supportsAllDrives=True),
                                     chunksize=50 * 1024 * 1024)
            done = False
            while not done:
                _, done = dl.next_chunk()
    return dst

def ocr_book(pdf):
    # OCR in separate processes, CHUNK_PAGES pages at a time; returns (text, pages)
    n = len(PdfReader(pdf).pages)
    step = CHUNK_PAGES if CHUNK_PAGES > 0 else n
    parts, deadline = [], time.time() + BOOK_TIMEOUT_MIN * 60
    for start in range(0, n, step):
        rng = "%d-%d" % (start, min(start + step, n) - 1) if step < n else "all"
        out = os.path.join(PDF_DIR, "part-%05d.md" % start)
        left = deadline - time.time()
        if left <= 0:
            raise RuntimeError("timeout after %d min" % BOOK_TIMEOUT_MIN)
        r = subprocess.run(["python", WORKER, pdf, out, rng, "1" if FORCE_OCR else "0"],
                           capture_output=True, text=True, timeout=left)
        if r.returncode != 0:
            raise RuntimeError("worker failed on pages %s: %s" % (rng, (r.stderr or "")[-500:]))
        parts.append(open(out, encoding="utf-8").read())
        print("   pages %s done" % rng)
    return "\n\n".join(parts), n
print("worker ready")

In [ ]:
# ===== Process the books one at a time (resumable) =====
done_now = 0
for book in BOOKS:
    manifest = load_manifest()
    rec = manifest["books"].get(book["key"])
    rel = os.path.join(OUTPUT_DIR, out_name(book["name"]))
    if rec is None:
        # books OCR'd before the manifest existed count as done
        rec = {"name": book["name"], "output": rel, "attempts": 0,
               "status": "done" if os.path.exists(os.path.join(REPO_DIR, rel)) else "pending"}
        manifest["books"][book["key"]] = rec
    if rec["status"] == "done":
        continue
    if rec["status"] == "failed" and not RETRY_FAILED:
        continue
    if rec["attempts"] >= MAX_ATTEMPTS and not RETRY_FAILED:
        rec["status"] = "failed"
        rec.setdefault("error", "crashed or stopped %d times" % rec["attempts"])
        save_manifest(manifest)
        push([MANIFEST], "chore: OCR manifest (%s failed)" % book["name"])
        continue
    if MAX_BOOKS and done_now >= MAX_BOOKS:
        print("MAX_BOOKS reached; run again to continue")
        break

    # record the attempt first, so a book that crashes the runtime is not retried forever
    rec.update(status="running", attempts=rec["attempts"] + 1,
               updated=datetime.datetime.now().isoformat(timespec="seconds"))
    save_manifest(manifest)
    push([MANIFEST], "chore: OCR start %s" % book["name"])
    print("==>", book["name"], "(attempt %d)" % rec["attempts"])
    t0 = time.time()
    try:
        text, pages = ocr_book(fetch(book))
        header = "<!-- raw OCR by marker-pdf %s from %s on %s (pages: %d, force_ocr: %s). Unedited. -->\n\n" % (
            md.version("marker-pdf"), book["name"], datetime.date.today().isoformat(), pages, FORCE_OCR)
        with open(os.path.join(REPO_DIR, rel), "w", encoding="utf-8") as f:
            f.write(header + text)
        manifest = load_manifest()
        manifest["books"][book["key"]].update(status="done", output=rel, pages=pages, chars=len(text),
                                               seconds=int(time.time() - t0), error=None,
                                               updated=datetime.datetime.now().isoformat(timespec="seconds"))
        save_manifest(manifest)
        push([rel, MANIFEST], "chore: add raw OCR markdown (%s)" % book["name"])
        print("pushed %s (%d pages, %d chars, %.0f s)" % (rel, pages, len(text), time.time() - t0))
        done_now += 1
    except Exception as e:
        manifest = load_manifest()
        r = manifest["books"][book["key"]]
        r.update(status="failed" if r["attempts"] >= MAX_ATTEMPTS else "pending", error=str(e)[:500],
                 updated=datetime.datetime.now().isoformat(timespec="seconds"))
        save_manifest(manifest)
        push([MANIFEST], "chore: OCR manifest (%s: error)" % book["name"])
        print("   error:", str(e)[:300])
    finally:
        shutil.rmtree(PDF_DIR, ignore_errors=True)

m = load_manifest()["books"]
summary = {}
for b in m.values():
    summary[b["status"]] = summary.get(b["status"], 0) + 1
print("finished this run:", done_now, "| manifest:", summary)